# Kolam — Crossing Loop (2 and 3 Anchor Dots)

## The Core Crossing Rule

For a **crossing (figure-8) loop** between anchor dots, the rule at every crossing midpoint is:

> **Arrive from one diagonal → depart on the opposite diagonal.**

For 2 dots — A at `(1,1)`, B at `(1,3)`, crossing at `(1,2)`:

```
Pass 1 (↘):  top-A (0,1)  →  cross (1,2)  →  bottom-B (2,3)
Pass 2 (↙):  top-B (0,3)  →  cross (1,2)  →  bottom-A (2,1)
```

These two passes approach `(1,2)` from **perpendicular directions** → actual ✕ self-intersection.

```
     (0,1)top-A           (0,3)top-B
          \                 /
           \    crossing   /
            ✕─────(1,2)─✕
           /               \
          /                 \
    (2,1)bot-A           (2,3)bot-B
```

The full loop then connects via arcs around A's left side and B's right side.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'monospace'
print('Ready.')

---
## 1. Lattice Helpers

In [ ]:
def make_lattice(m, n):
    return [(li, lj) for li in range(2*m+1) for lj in range(2*n+1)]

def is_anchor(li, lj):
    return li % 2 == 1 and lj % 2 == 1

def lattice_to_canvas(li, lj, sp, origin):
    return (origin[0] + lj * sp, origin[1] + li * sp)

def to_canvas(seq, sp, origin):
    return [lattice_to_canvas(li, lj, sp, origin) for li, lj in seq]

print('Lattice helpers ready.')

---
## 2. Chained Quadratic Bézier Renderer

In [ ]:
def qbez(p0, ctrl, p1, n=60):
    t = np.linspace(0, 1, n)
    x = (1-t)**2*p0[0] + 2*(1-t)*t*ctrl[0] + t**2*p1[0]
    y = (1-t)**2*p0[1] + 2*(1-t)*t*ctrl[1] + t**2*p1[1]
    return np.stack([x, y], axis=1)

def build_chained_bezier(pts, n=60):
    """Mirrors the Kolam TypeScript engine exactly."""
    if len(pts) < 2:
        return np.array(pts)
    all_pts = [pts[0]]
    for i in range(1, len(pts) - 1):
        ctrl = np.array(pts[i])
        mid  = (ctrl + np.array(pts[i+1])) / 2
        all_pts.extend(qbez(np.array(all_pts[-1]), ctrl, mid, n)[1:].tolist())
    end = np.array(pts[-1])
    all_pts.extend(qbez(np.array(all_pts[-1]),
                        (np.array(all_pts[-1]) + end) / 2, end, n)[1:].tolist())
    return np.array(all_pts)

print('Renderer ready.')

---
## 3. Crossing Loop Builder (n dots)

### The algorithm

For `n` anchor dots in a row, the crossing loop has **n−1 crossing midpoints**.
Each crossing is between anchor `k` and anchor `k+1`, located at lattice `(1, 2k+2)`.

The stroke has 4 phases:

```
1. FORWARD zigzag  : top-A → cross₀↘ → bot-B → cross₁↗ → top-C → ...
2. RIGHT arc       : loop around the last anchor's right side
3. BACKWARD zigzag : returns through each crossing from the opposite diagonal
4. LEFT arc        : loop around A's left side, close
```

The zigzag alternates top ↔ bottom at each crossing, which is exactly what creates the **perpendicular crossing** — each crossing midpoint is approached once from ↘ and once from ↙.

In [ ]:
def build_crossing_loop(n_dots: int):
    """
    Build a crossing (figure-8 chain) loop for n_dots anchors in a row.

    Anchors at lattice (1, 2k+1)  for k = 0 .. n_dots-1
    Crossings at      (1, 2k+2)   for k = 0 .. n_dots-2

    Each crossing midpoint is visited twice from perpendicular directions:
      Pass 1 (forward , k even):  (0, 2k+1) → (1, 2k+2) → (2, 2k+3)   going ↘
      Pass 1 (forward , k odd ):  (2, 2k+1) → (1, 2k+2) → (0, 2k+3)   going ↗
      Pass 2 (backward, reverse): opposite diagonal → true ✕ crossing
    """
    stroke = []

    # ── Phase 1: forward zigzag ────────────────────────────────────────────────
    stroke.append((0, 1))                      # top of first anchor (A)

    for k in range(n_dots - 1):
        cross_lj = 2*k + 2
        stroke.append((1, cross_lj))           # crossing between anchor k and k+1
        # After crossing, row alternates: 2 (bottom) if k even, 0 (top) if k odd
        row = 2 if k % 2 == 0 else 0
        stroke.append((row, 2*k + 3))          # bottom or top of anchor k+1

    # ── Phase 2: right arc around last anchor ──────────────────────────────────
    last_lj  = 2*(n_dots - 1) + 1             # lattice col of last anchor
    last_row = stroke[-1][0]                   # which row we ended on
    stroke.append((1, last_lj + 1))            # right of last anchor
    stroke.append((2 - last_row, last_lj))     # flip to opposite row

    # ── Phase 3: backward zigzag (2nd pass through each crossing) ─────────────
    for k in range(n_dots - 2, -1, -1):
        cross_lj = 2*k + 2
        stroke.append((1, cross_lj))           # crossing (2nd pass — creates ✕)
        # Same row rule as forward, but landing on anchor k instead of k+1
        row = 2 if k % 2 == 0 else 0
        stroke.append((row, 2*k + 1))          # bottom or top of anchor k

    # ── Phase 4: left arc and close ───────────────────────────────────────────
    stroke.append((1, 0))                      # left of first anchor
    stroke.append((0, 1))                      # close to start

    return stroke


# ── Validate for 2..5 dots ────────────────────────────────────────────────────
for nd in [2, 3, 4, 5]:
    s = build_crossing_loop(nd)

    # All midpoints
    for pt in s:
        assert not is_anchor(*pt), f'{pt} is an anchor!'

    # Each crossing visited exactly twice → true ✕ intersection
    crosses = [(1, 2*k+2) for k in range(nd-1)]
    for cp in crosses:
        assert s.count(cp) == 2, f'Crossing {cp} visited {s.count(cp)} times'

    # Path is closed (first == last)
    assert s[0] == s[-1], 'Path not closed!'

    print(f'{nd} dots: {len(s):>3} waypoints | {nd-1} crossing(s) each visited 2×  '
          f'| closed ✓')

print()
print('All validated!')

---
## 4. Step-by-Step: 2-Dot Crossing Loop

In [ ]:
s2 = build_crossing_loop(2)

roles_2 = [
    'top-A  ─ start',
    '✕ cross (1,2)  pass 1 ↘  (from upper-left)',
    'bottom-B       ─ shot past B diagonally',
    'right-B        ─ arc around B right side',
    'top-B          ─ back above B',
    '✕ cross (1,2)  pass 2 ↙  (from upper-right)  ← X here!',
    'bottom-A       ─ arrived below A',
    'left-A         ─ arc around A left side',
    'top-A  ─ close',
]

print('2-Dot Crossing Loop — full waypoint list')
print('─' * 60)
print(f'{"#":<4} {"(li,lj)":<10}  Role')
print('─' * 60)
for i, (pt, role) in enumerate(zip(s2, roles_2)):
    mark = '★' if pt == (1,2) else ' '
    print(f' {mark}[{i}]  ({pt[0]},{pt[1]})     {role}')
print()
print('Pass 1 direction at (1,2):  (0,1)→(1,2)→(2,3)  =  ↘')
print('Pass 2 direction at (1,2):  (0,3)→(1,2)→(2,1)  =  ↙')
print('↘ and ↙ cross each other → true ✕ self-intersection')

In [ ]:
# ── Render 2-dot crossing loop ────────────────────────────────────────────────

SP2, OR2 = 120, (80, 90)
M2, N2   = 1, 2

lat2    = make_lattice(M2, N2)
albl2   = {(1,1):'A', (1,3):'B'}
cross2  = [(1,2)]

crv2_canvas = to_canvas(s2, SP2, OR2)
crv2        = build_chained_bezier(crv2_canvas, n=80)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0a0a14')

for ax, annotate in zip(axes, [True, False]):
    ax.set_facecolor('#0d0d1e')

    for li, lj in lat2:
        x, y = lattice_to_canvas(li, lj, SP2, OR2)
        if is_anchor(li, lj):
            ax.plot(x, y, 'o', color='#f0c040', markersize=26, zorder=5,
                    markeredgecolor='#ffd700', markeredgewidth=2)
        else:
            ax.plot(x, y, '.', color='#181830', markersize=5, zorder=1)

    for (ali, alj), lbl in albl2.items():
        ax.annotate(lbl, lattice_to_canvas(ali, alj, SP2, OR2),
                    ha='center', va='center', fontsize=13,
                    color='#1a1a2e', fontweight='bold', zorder=12)

    # curve + glow
    ax.plot(crv2[:, 0], crv2[:, 1], color='#ff6b9d', linewidth=3.2, zorder=8)
    ax.plot(crv2[:, 0], crv2[:, 1], color='#ff9dc6', linewidth=12, alpha=0.12, zorder=7)

    # crossing star
    cx, cy = lattice_to_canvas(1, 2, SP2, OR2)
    ax.plot(cx, cy, '*', color='#ffd700', markersize=24, zorder=11,
            markeredgecolor='white', markeredgewidth=1.2)

    if annotate:
        seen = set()
        for lpt, cpt in zip(s2, crv2_canvas):
            col = '#ffd700' if lpt in cross2 else '#7ec8e3'
            ax.plot(cpt[0], cpt[1], 'D', color=col, markersize=9,
                    zorder=10, markeredgecolor='white', markeredgewidth=0.8)
            if lpt not in seen:
                ax.annotate(f'({lpt[0]},{lpt[1]})', (cpt[0], cpt[1]),
                            textcoords='offset points', xytext=(9, 5),
                            fontsize=7.5, color=col)
                seen.add(lpt)

        # draw the two crossing diagonals explicitly
        p_top_a  = lattice_to_canvas(0, 1, SP2, OR2)
        p_bot_b  = lattice_to_canvas(2, 3, SP2, OR2)
        p_top_b  = lattice_to_canvas(0, 3, SP2, OR2)
        p_bot_a  = lattice_to_canvas(2, 1, SP2, OR2)

        ax.annotate('', xy=p_bot_b, xytext=p_top_a,
            arrowprops=dict(arrowstyle='->', color='#48dbfb',
                            lw=1.5, linestyle='dashed'))
        ax.annotate('pass 1 ↘', p_top_a,
            textcoords='offset points', xytext=(-50, 5),
            fontsize=8, color='#48dbfb')

        ax.annotate('', xy=p_bot_a, xytext=p_top_b,
            arrowprops=dict(arrowstyle='->', color='#ff9f43',
                            lw=1.5, linestyle='dashed'))
        ax.annotate('pass 2 ↙', p_top_b,
            textcoords='offset points', xytext=(10, 5),
            fontsize=8, color='#ff9f43')

        ax.annotate('✕\n(1,2)', (cx, cy),
            textcoords='offset points', xytext=(14, -24),
            fontsize=9, color='#ffd700')

        ax.set_title('Annotated — two diagonal passes cross at (1,2)', color='white', fontsize=9)
    else:
        ax.set_title('Clean render — 2-dot crossing loop', color='white', fontsize=10)

    ax.set_xlim(OR2[0]-SP2*0.9, OR2[0]+(2*N2+1)*SP2+SP2*0.5)
    ax.set_ylim(OR2[1]-SP2*1.1, OR2[1]+(2*M2+1)*SP2+SP2*0.9)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(colors='#333')

axes[0].legend(handles=[
    Line2D([0],[0], color='#ff6b9d', lw=2.5, label='Crossing loop stroke'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#f0c040',
           markersize=10, label='Anchor dot (never touched)', linewidth=0),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#ffd700',
           markersize=12, label='Self-intersection (1,2)', linewidth=0),
    Line2D([0],[0], color='#48dbfb', lw=1.5, ls='--', label='Pass 1 ↘  (top-A → bot-B)'),
    Line2D([0],[0], color='#ff9f43', lw=1.5, ls='--', label='Pass 2 ↙  (top-B → bot-A)'),
], loc='lower right', facecolor='#0a0a14', edgecolor='#333',
   labelcolor='white', fontsize=7.5)

fig.suptitle('2-Dot Crossing Loop — True ✕ at (1,2)',
             color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Step-by-Step: 3-Dot Crossing Loop

Two crossings: one between A–B at `(1,2)`, one between B–C at `(1,4)`.

```
Forward zigzag:
  top-A (0,1)  →  ✕(1,2)↘  →  bot-B (2,3)  →  ✕(1,4)↗  →  top-C (0,5)

Right arc around C:  top-C → right-C (1,6) → bot-C (2,5)

Backward zigzag:
  bot-C (2,5)  →  ✕(1,4)↖  →  top-B (0,3)  →  ✕(1,2)↙  →  bot-A (2,1)

Left arc around A:  bot-A → left-A (1,0) → top-A (0,1)  close
```

In [ ]:
s3 = build_crossing_loop(3)

roles_3 = [
    'top-A  ─ start',
    '✕ cross-AB (1,2)  pass 1 ↘',
    'bottom-B   ─ shot past B diagonally',
    '✕ cross-BC (1,4)  pass 1 ↗',
    'top-C      ─ arrived above C',
    'right-C    ─ arc around C',
    'bottom-C',
    '✕ cross-BC (1,4)  pass 2 ↖  ← ✕ here!',
    'top-B      ─ arrived above B ("immediately go down" next)',
    '✕ cross-AB (1,2)  pass 2 ↙  ← ✕ here!',
    'bottom-A',
    'left-A     ─ arc around A',
    'top-A  ─ close',
]

print('3-Dot Crossing Loop — full waypoint list')
print('─' * 65)
print(f'{"#":<5} {"(li,lj)":<10}  Role')
print('─' * 65)
crosses3 = [(1,2),(1,4)]
for i, (pt, role) in enumerate(zip(s3, roles_3)):
    mark = '★' if pt in crosses3 else ' '
    print(f' {mark}[{i:>2}]  ({pt[0]},{pt[1]})     {role}')

print()
print('Cross-AB (1,2): pass1 direction (0,1)→(1,2)→(2,3) = ↘')
print('               pass2 direction (0,3)→(1,2)→(2,1) = ↙  → ✕')
print()
print('Cross-BC (1,4): pass1 direction (2,3)→(1,4)→(0,5) = ↗')
print('               pass2 direction (2,5)→(1,4)→(0,3) = ↖  → ✕')

In [ ]:
# ── Render 3-dot crossing loop ────────────────────────────────────────────────

SP3, OR3 = 105, (65, 85)
M3, N3   = 1, 3

lat3     = make_lattice(M3, N3)
albl3    = {(1,1):'A', (1,3):'B', (1,5):'C'}
crosses3 = [(1,2), (1,4)]

crv3_canvas = to_canvas(s3, SP3, OR3)
crv3        = build_chained_bezier(crv3_canvas, n=80)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0a0a14')

for ax, annotate in zip(axes, [True, False]):
    ax.set_facecolor('#0d0d1e')

    for li, lj in lat3:
        x, y = lattice_to_canvas(li, lj, SP3, OR3)
        if is_anchor(li, lj):
            ax.plot(x, y, 'o', color='#f0c040', markersize=24, zorder=5,
                    markeredgecolor='#ffd700', markeredgewidth=2)
        else:
            ax.plot(x, y, '.', color='#181830', markersize=5, zorder=1)

    for (ali, alj), lbl in albl3.items():
        ax.annotate(lbl, lattice_to_canvas(ali, alj, SP3, OR3),
                    ha='center', va='center', fontsize=12,
                    color='#1a1a2e', fontweight='bold', zorder=12)

    ax.plot(crv3[:, 0], crv3[:, 1], color='#ff6b9d', linewidth=3.0, zorder=8)
    ax.plot(crv3[:, 0], crv3[:, 1], color='#ff9dc6', linewidth=12, alpha=0.12, zorder=7)

    for cp in crosses3:
        cx, cy = lattice_to_canvas(*cp, SP3, OR3)
        ax.plot(cx, cy, '*', color='#ffd700', markersize=22, zorder=11,
                markeredgecolor='white', markeredgewidth=1.1)

    if annotate:
        seen = set()
        for lpt, cpt in zip(s3, crv3_canvas):
            col = '#ffd700' if lpt in crosses3 else '#7ec8e3'
            ax.plot(cpt[0], cpt[1], 'D', color=col, markersize=8,
                    zorder=10, markeredgecolor='white', markeredgewidth=0.8)
            if lpt not in seen:
                ax.annotate(f'({lpt[0]},{lpt[1]})', (cpt[0], cpt[1]),
                            textcoords='offset points', xytext=(8, 5),
                            fontsize=7, color=col)
                seen.add(lpt)

        # draw the 4 crossing diagonals
        diag_pairs = [
            ((0,1),(2,3),'#48dbfb','AB pass1 ↘'),
            ((0,3),(2,1),'#ff9f43','AB pass2 ↙'),
            ((2,3),(0,5),'#a29bfe','BC pass1 ↗'),
            ((2,5),(0,3),'#fd79a8','BC pass2 ↖'),
        ]
        for (pa, pb, col, label) in diag_pairs:
            ca = lattice_to_canvas(*pa, SP3, OR3)
            cb = lattice_to_canvas(*pb, SP3, OR3)
            ax.annotate('', xy=cb, xytext=ca,
                arrowprops=dict(arrowstyle='->', color=col,
                                lw=1.4, linestyle='dashed'))

        for cp, name in zip(crosses3, ['✕ AB\n(1,2)','✕ BC\n(1,4)']):
            cx, cy = lattice_to_canvas(*cp, SP3, OR3)
            ax.annotate(name, (cx, cy),
                textcoords='offset points', xytext=(12, -28),
                fontsize=8, color='#ffd700')

        ax.set_title('Annotated — 4 diagonal passes, 2 crossings', color='white', fontsize=9)
    else:
        ax.set_title('Clean render — 3-dot crossing loop', color='white', fontsize=10)

    ax.set_xlim(OR3[0]-SP3*0.9, OR3[0]+(2*N3+1)*SP3+SP3*0.5)
    ax.set_ylim(OR3[1]-SP3*1.1, OR3[1]+(2*M3+1)*SP3+SP3*0.9)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(colors='#333')

axes[0].legend(handles=[
    Line2D([0],[0], color='#ff6b9d', lw=2.5, label='Single closed stroke'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#ffd700',
           markersize=11, label='Self-intersections', linewidth=0),
    Line2D([0],[0], color='#48dbfb', lw=1.4, ls='--', label='AB pass1 ↘'),
    Line2D([0],[0], color='#ff9f43', lw=1.4, ls='--', label='AB pass2 ↙'),
    Line2D([0],[0], color='#a29bfe', lw=1.4, ls='--', label='BC pass1 ↗'),
    Line2D([0],[0], color='#fd79a8', lw=1.4, ls='--', label='BC pass2 ↖'),
], loc='lower right', facecolor='#0a0a14', edgecolor='#333',
   labelcolor='white', fontsize=7.5)

fig.suptitle('3-Dot Crossing Loop — ✕ at (1,2) and ✕ at (1,4)',
             color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Scale-Up: 2 → 3 → 4 → 5 Dots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#0a0a14')
axes = axes.flatten()

for ax, nd in zip(axes, [2, 3, 4, 5]):
    ax.set_facecolor('#0d0d1e')
    sp  = max(50, 115 - nd * 13)
    or_ = (50, 65)
    lat = make_lattice(1, nd)

    stroke = build_crossing_loop(nd)
    crv    = build_chained_bezier(to_canvas(stroke, sp, or_), n=80)

    for li, lj in lat:
        x, y = lattice_to_canvas(li, lj, sp, or_)
        if is_anchor(li, lj):
            ax.plot(x, y, 'o', color='#f0c040', markersize=17, zorder=5,
                    markeredgecolor='#ffd700', markeredgewidth=1.5)
            k = (lj-1)//2
            ax.annotate(chr(ord('A')+k), (x, y), ha='center', va='center',
                        fontsize=9, color='#1a1a2e', fontweight='bold', zorder=12)
        else:
            ax.plot(x, y, '.', color='#181830', markersize=4, zorder=1)

    ax.plot(crv[:, 0], crv[:, 1], color='#ff6b9d', linewidth=2.8, zorder=8)
    ax.plot(crv[:, 0], crv[:, 1], color='#ff9dc6', linewidth=10, alpha=0.12, zorder=7)

    for k in range(nd-1):
        cp = (1, 2*k+2)
        cx, cy = lattice_to_canvas(*cp, sp, or_)
        ax.plot(cx, cy, '*', color='#ffd700', markersize=14, zorder=11,
                markeredgecolor='white', markeredgewidth=0.8)

    cols = 2*nd+1
    ax.set_xlim(or_[0]-sp*0.9, or_[0]+cols*sp+sp*0.5)
    ax.set_ylim(or_[1]-sp*1.0, or_[1]+3*sp+sp*0.9)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(f'{nd} dots  —  {nd-1} crossing{"s" if nd>2 else ""}  '
                 f'({len(stroke)} waypoints)',
                 color='white', fontsize=10)
    for sp_ in ax.spines.values(): sp_.set_visible(False)
    ax.tick_params(colors='#333')

fig.suptitle('Crossing Loop Scaled: 2 → 3 → 4 → 5 Anchor Dots  (★ = self-intersection)',
             color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. The Crossing Rule — Summary

At every crossing midpoint `(1, 2k+2)` between anchor `k` and `k+1`:

| Pass | Coming from | Going to | Direction |
|---|---|---|---|
| Forward (k even) | top of anchor k `(0, 2k+1)` | bottom of anchor k+1 `(2, 2k+3)` | ↘ |
| Forward (k odd)  | bottom of anchor k `(2, 2k+1)` | top of anchor k+1 `(0, 2k+3)` | ↗ |
| Backward (k even)| top of anchor k+1 `(0, 2k+3)` | bottom of anchor k `(2, 2k+1)` | ↙ |
| Backward (k odd) | bottom of anchor k+1 `(2, 2k+3)` | top of anchor k `(0, 2k+1)` | ↖ |

The forward and backward passes through the same crossing are **always perpendicular** → guaranteed ✕.

```
n dots  →  n-1 crossings  →  n-1 self-intersections  →  n sub-loops
```

In [ ]:
import json

sample = {
    "image": "kolam_crossing_3dot.png",
    "grid": {"rows": 1, "cols": 3},
    "hypergraph": {
        "vertices": ["A", "B", "C"],
        "vertex_lattice": {"A": [1,1], "B": [1,3], "C": [1,5]},
        "relations": [{
            "type": "NORMAL_LOOP",
            "vertices": ["A", "B", "C"],
            "crossing_midpoints": [[1,2], [1,4]],
            "crossing_rule": "forward pass ↘/↗ zigzag, backward pass ↙/↖ zigzag — perpendicular at each crossing → true ✕"
        }]
    },
    "compiled_lattice_sequence": [
        [0,1],        # top-A  start
        [1,2],        # ✕ cross-AB  pass1 ↘
        [2,3],        # bot-B
        [1,4],        # ✕ cross-BC  pass1 ↗
        [0,5],        # top-C
        [1,6],        # right-C  arc
        [2,5],        # bot-C
        [1,4],        # ✕ cross-BC  pass2 ↖
        [0,3],        # top-B  (arrived — immediately cross down next)
        [1,2],        # ✕ cross-AB  pass2 ↙
        [2,1],        # bot-A
        [1,0],        # left-A  arc
        [0,1],        # close
    ]
}

print(json.dumps(sample, indent=2))